# 03 - Layer A Text Retrieval Debug

Notebook này dùng để soi riêng nhánh **tìm sản phẩm bằng text**.

Góc nhìn cần giữ trong đầu:

```text
query tiếng Việt
  -> ViFashionCLIPTextEmbeddings
  -> Qdrant lấy raw candidates
  -> optional reranker chấm lại
  -> diversity filter giảm trùng sản phẩm/brand
  -> final docs gửi cho LLM
```

Notebook này **không debug prompt, không debug chat loop, không debug Layer B phối đồ**. Nếu final docs đã sai, đừng sửa prompt vội; hãy sửa retrieval trước.

Bạn có thể mở notebook này độc lập. Nó import class/config trực tiếp từ thư mục `app/`, không cần chạy notebook 01 trước.

Notebook 01 vẫn nên đọc trước nếu bạn muốn hiểu mô hình embedding được khai báo như thế nào.


## BƯỚC 1: Setup Tối Thiểu

Cell này tìm thư mục gốc `Chatbot_Fashion/`, thêm nó vào `sys.path`, rồi import thẳng từ code app.

Cách này tốt hơn việc phụ thuộc vào “đã chạy notebook 01 chưa”, vì mỗi notebook có thể chạy độc lập và ít lỗi thiếu biến hơn.
Import model embedding được để sang bước 3 để cell setup không bị chậm.


In [1]:
import sys
from pathlib import Path

from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


def find_chatbot_fashion_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "app" / "config.py").exists():
            return candidate
    raise RuntimeError("Không tìm thấy thư mục gốc Chatbot_Fashion chứa app/config.py")


CHATBOT_FASHION_DIR = find_chatbot_fashion_root()
if str(CHATBOT_FASHION_DIR) not in sys.path:
    sys.path.insert(0, str(CHATBOT_FASHION_DIR))

from app.config import (
    PRODUCT_SEARCH_BRAND_LIMIT as APP_PRODUCT_SEARCH_BRAND_LIMIT,
    PRODUCT_SEARCH_CANDIDATE_K as APP_PRODUCT_SEARCH_CANDIDATE_K,
    PRODUCT_SEARCH_PAGE_SIZE as APP_PRODUCT_SEARCH_PAGE_SIZE,
    QDRANT_COLLECTION_FASHION,
    QDRANT_URL as APP_QDRANT_URL,
    RERANKER_BATCH_SIZE as APP_RERANKER_BATCH_SIZE,
    RERANKER_MODEL_NAME as APP_RERANKER_MODEL_NAME,
    RERANKER_TOP_N as APP_RERANKER_TOP_N,
)

print("[OK] Setup notebook 03 hoàn tất")
print(f"Project root: {CHATBOT_FASHION_DIR}")


[OK] Setup notebook 03 hoàn tất
Project root: D:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion


## BƯỚC 2: Cấu Hình Retrieval

Các số dưới đây ảnh hưởng trực tiếp đến chất lượng và tốc độ:

- `PRODUCT_SEARCH_CANDIDATE_K`: lấy bao nhiêu ứng viên thô từ Qdrant.
- `RERANKER_TOP_N`: rerank bao nhiêu ứng viên đầu.
- `PRODUCT_SEARCH_PAGE_SIZE`: cuối cùng giữ bao nhiêu sản phẩm.
- `ENABLE_PRODUCT_RERANKER`: nên để `False` khi mới debug raw retrieval; bật `True` sau khi raw candidates đã có sản phẩm đúng.


In [2]:
QDRANT_URL = APP_QDRANT_URL
PRODUCT_COLLECTION = QDRANT_COLLECTION_FASHION

PRODUCT_SEARCH_CANDIDATE_K = APP_PRODUCT_SEARCH_CANDIDATE_K
PRODUCT_SEARCH_PAGE_SIZE = APP_PRODUCT_SEARCH_PAGE_SIZE
PRODUCT_SEARCH_BRAND_LIMIT = APP_PRODUCT_SEARCH_BRAND_LIMIT

# Để debug nhanh, mặc định tắt reranker.
# Khi muốn so sánh chất lượng sau dense retrieval, đổi thành True.
ENABLE_PRODUCT_RERANKER = True
RERANKER_MODEL_NAME = APP_RERANKER_MODEL_NAME
RERANKER_TOP_N = APP_RERANKER_TOP_N
RERANKER_BATCH_SIZE = APP_RERANKER_BATCH_SIZE

print("[OK] Retrieval config")
print(f"Collection : {PRODUCT_COLLECTION}")
flow = f"top-{PRODUCT_SEARCH_CANDIDATE_K} raw"
if ENABLE_PRODUCT_RERANKER:
    flow += f" -> rerank top-{RERANKER_TOP_N}"
flow += f" -> diversity -> final {PRODUCT_SEARCH_PAGE_SIZE}"
print(f"Flow       : {flow}")
print(f"Reranker   : {ENABLE_PRODUCT_RERANKER}")


[OK] Retrieval config
Collection : fashion_products_vifashionclip_vi_65k_structured_vi
Flow       : top-15 raw -> rerank top-8 -> diversity -> final 5
Reranker   : True


## BƯỚC 3: Kết Nối Qdrant Và Embedding

Cell này là cell nặng nhất của notebook 03 vì nó khởi tạo `ViFashionCLIPTextEmbeddings`.

Nếu chạy chậm hoặc lỗi:

- Qdrant có thể chưa bật ở `localhost:6333`.
- ViFashionCLIP checkpoint/model có thể load chậm.
- Nếu `PRODUCT_EMBEDDING_BACKEND=remote`, notebook sẽ gọi embedding service giống app.


In [3]:
print("[INFO] Khởi tạo ViFashionCLIP text embedding...")
from app.core.embeddings import get_product_embeddings

product_embeddings = get_product_embeddings()

print("[INFO] Kết nối Qdrant...")
client = QdrantClient(url=QDRANT_URL, timeout=20, check_compatibility=False)

if not client.collection_exists(PRODUCT_COLLECTION):
    raise RuntimeError(f"Không thấy collection Qdrant: {PRODUCT_COLLECTION}")

count = client.count(PRODUCT_COLLECTION).count
print(f"[OK] Collection `{PRODUCT_COLLECTION}` có {count} points")

vector_db = QdrantVectorStore(
    client=client,
    collection_name=PRODUCT_COLLECTION,
    embedding=product_embeddings,
)
print("[OK] vector_db sẵn sàng")


[INFO] Khởi tạo ViFashionCLIP text embedding...
[INFO] Using remote ViFashionCLIP embedding service: http://localhost:18080
[INFO] Kết nối Qdrant...
[OK] Collection `fashion_products_vifashionclip_vi_65k_structured_vi` có 65480 points
[OK] vector_db sẵn sàng


## BƯỚC 4: Chuẩn Hóa Metadata Và Lọc Đa Dạng

Hai hàm này xử lý phần sau retrieval:

- `normalize_product_metadata`: đảm bảo document có `image_url`.
- `diversity_filter_documents`: giảm lặp cùng `product_id` hoặc cùng brand quá nhiều.

Nếu bạn thấy chatbot gợi ý nhiều món giống nhau, hãy nhìn hàm diversity này.


In [4]:
def normalize_product_metadata(doc: Document) -> Document:
    """Đảm bảo mỗi Document có `images` dạng list và `image_url` ổn định."""
    images = doc.metadata.get("images", [])
    if isinstance(images, str):
        images = [images] if images else []
    doc.metadata["images"] = images
    doc.metadata["image_url"] = doc.metadata.get("image_url") or (images[0] if images else "")
    return doc


def diversity_filter_documents(
    docs: list[Document],
    max_docs: int = PRODUCT_SEARCH_PAGE_SIZE,
    max_per_brand: int = PRODUCT_SEARCH_BRAND_LIMIT,
) -> list[Document]:
    """Lọc trùng product_id và giới hạn số sản phẩm cùng brand."""
    selected: list[Document] = []
    seen_product_ids: set[str] = set()
    brand_counts: dict[str, int] = {}

    for doc in docs:
        doc = normalize_product_metadata(doc)
        product_id = str(doc.metadata.get("product_id", "")).strip().lower()
        brand = str(doc.metadata.get("brand", "")).strip().lower()

        if product_id and product_id in seen_product_ids:
            continue
        if brand and brand_counts.get(brand, 0) >= max_per_brand:
            continue

        selected.append(doc)
        if product_id:
            seen_product_ids.add(product_id)
        if brand:
            brand_counts[brand] = brand_counts.get(brand, 0) + 1
        if len(selected) >= max_docs:
            return selected

    return selected


## BƯỚC 5: Dense Retrieval - Xem Raw Candidates Từ Qdrant

Đây là bước quan trọng nhất.

Nếu top-30 raw candidates đã không có sản phẩm đúng, vấn đề thường nằm ở:

- query đưa vào chưa đúng ý;
- `page_content` sản phẩm thiếu tín hiệu;
- embedding chưa tốt;
- collection Qdrant không đúng dữ liệu.


In [5]:
def extract_labeled_field(page_content: str, label: str, default: str = "") -> str:
    """Read one labeled value from product page_content, only for display."""
    prefix = f"{label}:"
    for line in (page_content or "").splitlines():
        line = line.strip()
        if line.startswith(prefix):
            return line.split(":", 1)[1].strip() or default
    return default


def doc_row(doc: Document, rank: int) -> dict:
    """Compact one Document into a debug-friendly row."""
    doc = normalize_product_metadata(doc)
    return {
        "rank": rank,
        "product_id": doc.metadata.get("product_id", ""),
        "title": doc.metadata.get("title", ""),
        "category": doc.metadata.get("category", ""),
        "brand": doc.metadata.get("brand", ""),
        "price": doc.metadata.get("price", "") or extract_labeled_field(doc.page_content, "Giá"),
        "target": doc.metadata.get("department", "") or extract_labeled_field(doc.page_content, "Đối tượng"),
        "color": doc.metadata.get("main_color", "") or extract_labeled_field(doc.page_content, "Màu sắc"),
        "material": doc.metadata.get("material", "") or extract_labeled_field(doc.page_content, "Chất liệu"),
        "occasion": doc.metadata.get("occasion", "") or extract_labeled_field(doc.page_content, "Dịp sử dụng"),
        "dense_score": doc.metadata.get("dense_score"),
        "rerank_score": doc.metadata.get("rerank_score"),
        # "preview": doc.page_content[:180].replace("\n", " | "),
    }


def print_doc_rows(title: str, docs: list[Document], limit: int = 10) -> None:
    print(f"\n=== {title} ({len(docs)}) ===")
    for rank, doc in enumerate(docs[:limit], start=1):
        row = doc_row(doc, rank)
        score = row["dense_score"]
        rerank = row["rerank_score"]
        score_text = "" if score is None else f" dense={float(score):.4f}"
        rerank_text = "" if rerank is None else f" rerank={float(rerank):.4f}"
        print(f"#{rank}{score_text}{rerank_text} | {row['product_id']} | {row['category']} | {row['brand']}")
        print(
            "  "
            f"price={row['price']} | target={row['target']} | color={row['color']} | "
            f"material={row['material']} | occasion={row['occasion']}"
        )
        print(f"  title: {row['title']}")
        # print(f"  preview: {row['preview']}")


def retrieve_raw_candidates(query: str, k: int = PRODUCT_SEARCH_CANDIDATE_K) -> list[Document]:
    """Call Qdrant similarity search and attach dense_score/dense_rank metadata."""
    pairs = vector_db.similarity_search_with_score(query=query, k=k)
    docs: list[Document] = []
    for rank, (doc, score) in enumerate(pairs, start=1):
        doc = normalize_product_metadata(doc)
        doc.metadata["dense_rank"] = rank
        doc.metadata["dense_score"] = float(score)
        docs.append(doc)
    return docs


## BƯỚC 6: Optional Reranker

Reranker đọc cặp `(query, product text)` kỹ hơn dense retrieval, nhưng chậm hơn.

Khi mới debug, hãy xem raw candidates trước. Chỉ bật reranker khi bạn muốn so sánh:

```python
ENABLE_PRODUCT_RERANKER = True
```


In [6]:
product_reranker = None


class ProductCrossEncoderReranker:
    """Cross-encoder reranker, chỉ load khi `ENABLE_PRODUCT_RERANKER=True`."""

    def __init__(self, model_name: str = RERANKER_MODEL_NAME):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.torch = torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device).eval()
        print(f"[OK] Reranker loaded: {model_name} on {self.device}")

    def score_pairs(self, query: str, docs: list[Document]) -> list[float]:
        pairs = [(query, doc.page_content[:1400]) for doc in docs]
        scores: list[float] = []
        with self.torch.no_grad():
            for start in range(0, len(pairs), RERANKER_BATCH_SIZE):
                batch = pairs[start:start + RERANKER_BATCH_SIZE]
                inputs = self.tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt",
                ).to(self.device)
                logits = self.model(**inputs).logits
                batch_scores = logits[:, -1] if logits.ndim == 2 and logits.shape[-1] > 1 else logits.reshape(-1)
                scores.extend(batch_scores.detach().float().cpu().tolist())
        return scores


def get_product_reranker():
    global product_reranker, ENABLE_PRODUCT_RERANKER
    if not ENABLE_PRODUCT_RERANKER:
        return None
    if product_reranker is None:
        try:
            product_reranker = ProductCrossEncoderReranker()
        except Exception as exc:
            ENABLE_PRODUCT_RERANKER = False
            print(f"[WARN] Không load được reranker, dùng dense order: {exc}")
            return None
    return product_reranker


def rerank_documents(query: str, docs: list[Document], top_n: int = RERANKER_TOP_N) -> list[Document]:
    reranker = get_product_reranker()
    if reranker is None or not docs:
        return docs

    scores = reranker.score_pairs(query, docs[:top_n])
    ranked = sorted(zip(docs[:top_n], scores), key=lambda pair: pair[1], reverse=True)
    output: list[Document] = []
    for rank, (doc, score) in enumerate(ranked, start=1):
        doc.metadata["rerank_rank"] = rank
        doc.metadata["rerank_score"] = float(score)
        output.append(doc)
    return output


## BƯỚC 7: Debug Một Query Từ Đầu Đến Cuối

Hàm này in ba lát cắt:

1. `Raw Qdrant candidates`: Qdrant lấy gì.
2. `After rerank`: reranker có đổi thứ tự không.
3. `Final docs`: các sản phẩm thật sự sẽ gửi vào LLM.


In [7]:
def debug_layer_a_query(query: str, k: int = PRODUCT_SEARCH_CANDIDATE_K) -> dict[str, list[Document]]:
    import time

    print(f"QUERY: {query}")
    total_start = time.perf_counter()

    raw_start = time.perf_counter()
    raw_docs = retrieve_raw_candidates(query, k=k)
    raw_seconds = time.perf_counter() - raw_start
    print_doc_rows("Raw Qdrant candidates", raw_docs, limit=min(10, k))

    rerank_start = time.perf_counter()
    ranked_docs = rerank_documents(query, raw_docs, top_n=RERANKER_TOP_N)
    rerank_seconds = time.perf_counter() - rerank_start
    print_doc_rows("After optional rerank", ranked_docs, limit=min(10, len(ranked_docs)))

    final_start = time.perf_counter()
    final_docs = diversity_filter_documents(ranked_docs, max_docs=PRODUCT_SEARCH_PAGE_SIZE)
    final_seconds = time.perf_counter() - final_start
    print_doc_rows("Final docs sent to LLM", final_docs, limit=PRODUCT_SEARCH_PAGE_SIZE)

    total_seconds = time.perf_counter() - total_start
    timings = {
        "raw_retrieval_seconds": raw_seconds,
        "rerank_seconds": rerank_seconds,
        "diversity_final_seconds": final_seconds,
        "total_seconds": total_seconds,
    }

    print()
    print("=== Timing ===")
    print(f"raw retrieval : {raw_seconds:.3f}s")
    print(f"rerank        : {rerank_seconds:.3f}s")
    print(f"diversity     : {final_seconds:.3f}s")
    print(f"total         : {total_seconds:.3f}s")

    return {
        "raw_docs": raw_docs,
        "ranked_docs": ranked_docs,
        "final_docs": final_docs,
        "timings": timings,
    }


In [9]:
TEST_QUERIES = [
    "tìm áo sơ mi trắng nam",
    "cho tôi xem váy đen đi tiệc",
    "tìm áo thun nữ cotton giá dưới 300k",
    "tìm giày sneaker nam màu trắng dưới 700k",
    "tìm áo khoác denim nam Tokisakail đi chơi",
    "tìm áo sơ mi dài tay công sở nhưng không phải màu trắng",
    "tôi muốn váy đi tiệc màu đen hoặc xanh đậm, đừng lấy màu trắng",
    "tìm áo nữ mặc đi làm, lịch sự, giá dưới 500k, ưu tiên màu be hoặc kem",
    "có quần đi chơi cho nam không quá đắt, tránh màu trắng và màu hồng không",
    "tôi cần đồ đi phỏng vấn nhìn lịch sự nhưng không quá formal, không màu trắng, ngân sách khoảng dưới 800k",
]

# Đổi số này từ 0 đến 9 để soi từng câu
QUERY_INDEX = 4

selected_query = TEST_QUERIES[QUERY_INDEX]
result = debug_layer_a_query(selected_query)


QUERY: tìm áo khoác denim nam Tokisakail đi chơi

=== Raw Qdrant candidates (15) ===
#1 dense=0.7180 | B07MZXCPXX | Áo khoác | Siawasey
  price=813120 | target=Unisex | color=Denim | material=Không rõ | occasion=Học đường, Xã hội, Hàng ngày
  title: Áo khoác hoodie denim Siawasey Anime
#2 dense=0.7103 | B0912Y5V23 | Áo khoác | Tokisaki
  price=887760 | target=Unisex | color=Không rõ | material=Cotton, Polyester, Rayon, Spandex | occasion=Xã hội, Hàng ngày
  title: Áo khoác denim có mũ in hình anime
#3 dense=0.7053 | B08ZLLY17Q | Áo khoác | Tokisaki
  price=887760 | target=Unisex | color=Không rõ | material=Polyester, Rayon, Spandex | occasion=Xã hội, Hàng ngày
  title: Áo khoác denim nam nữ Tokyo Ghoul rách có mũ
#4 dense=0.6850 | B07YK4Z5C3 | Áo khoác | NINE WEST
  price=1439760 | target=Nữ | color=Không rõ | material=29% Polyester | occasion=Hàng ngày
  title: Áo khoác denim Nine West Sarah - Bengal Bananza
#5 dense=0.6841 | B00SVIKPGC | Áo khoác | Milwaukee
  price=1631759 | target=